In essence, we cannot “solve” instances of LWE. But estimating how hard they are is slightly more studied. I think going for a solver is way-way beyond my scope, thus https://github.com/malb/lattice-estimator by Albrecht et al will come in handy.

In [1]:
from sage.crypto.lwe import samples

In [2]:
S = samples(4, 4, 'Regev')
S

[((9, 8, 13, 12), 14),
 ((16, 16, 12, 5), 7),
 ((0, 3, 5, 12), 14),
 ((5, 8, 9, 12), 4)]

In [3]:
# importing submodules with hyphens: https://stackoverflow.com/questions/8350853/
import sys
sys.path.append('lattice-estimator')
import estimator

In [4]:
estimator.LWE.Estimate.rough

<function Estimate.rough at 0x7f9f18109f30>

## Lattice estimator usage with built-in parameter sets

In [5]:
%%time

estimator.schemes.Kyber512

CPU times: user 5 μs, sys: 1 μs, total: 6 μs
Wall time: 8.58 μs


LWEParameters(n=512, q=3329, Xs=D(σ=1.22), Xe=D(σ=1.22), m=512, tag='Kyber 512')

In [6]:
%%time
estimator.LWE.primal_usvp(estimator.schemes.Kyber512)

CPU times: user 94 ms, sys: 3.85 ms, total: 97.8 ms
Wall time: 97.4 ms


rop: ≈2^143.8, red: ≈2^143.8, δ: 1.003941, β: 406, d: 998, tag: usvp

In [7]:
%%time
estimator.LWE.estimate.rough(estimator.schemes.Kyber512)

usvp                 :: rop: ≈2^118.6, red: ≈2^118.6, δ: 1.003941, β: 406, d: 998, tag: usvp
dual_hybrid          :: rop: ≈2^115.5, red: ≈2^115.3, guess: ≈2^112.3, β: 395, p: 5, ζ: 0, t: 40, β': 395, N: ≈2^81.4, m: 512
CPU times: user 3.33 s, sys: 7.32 ms, total: 3.34 s
Wall time: 3.36 s


{'usvp': rop: ≈2^118.6, red: ≈2^118.6, δ: 1.003941, β: 406, d: 998, tag: usvp,
 'dual_hybrid': rop: ≈2^115.5, red: ≈2^115.3, guess: ≈2^112.3, β: 395, p: 5, ζ: 0, t: 40, β': 395, N: ≈2^81.4, m: 512}

In [8]:
%%time
estimator.LWE.estimate(estimator.schemes.Kyber512)

bkw                  :: rop: ≈2^178.8, m: ≈2^166.8, mem: ≈2^167.8, b: 14, t1: 0, t2: 16, ℓ: 13, #cod: 448, #top: 0, #test: 64, tag: coded-bkw
usvp                 :: rop: ≈2^143.8, red: ≈2^143.8, δ: 1.003941, β: 406, d: 998, tag: usvp
bdd                  :: rop: ≈2^140.2, red: ≈2^139.1, svp: ≈2^139.3, β: 389, η: 422, d: 1005, tag: bdd
dual                 :: rop: ≈2^149.9, mem: ≈2^97.1, m: 512, β: 424, d: 1024, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^139.7, red: ≈2^139.5, guess: ≈2^135.9, β: 387, p: 5, ζ: 0, t: 50, β': 391, N: ≈2^81.1, m: 512
CPU times: user 25.8 s, sys: 78.1 ms, total: 25.9 s
Wall time: 26.4 s


{'arora-gb': rop: ≈2^inf,
 'bkw': rop: ≈2^178.8, m: ≈2^166.8, mem: ≈2^167.8, b: 14, t1: 0, t2: 16, ℓ: 13, #cod: 448, #top: 0, #test: 64, tag: coded-bkw,
 'usvp': rop: ≈2^143.8, red: ≈2^143.8, δ: 1.003941, β: 406, d: 998, tag: usvp,
 'bdd': rop: ≈2^140.2, red: ≈2^139.1, svp: ≈2^139.3, β: 389, η: 422, d: 1005, tag: bdd,
 'bdd_hybrid': rop: ≈2^140.2, red: ≈2^139.1, svp: ≈2^139.3, β: 389, η: 422, ζ: 0, |S|: 1, d: 1013, prob: 1.0, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^262.1, red: ≈2^254.0, svp: ≈2^262.1, β: 407, η: 2, ζ: 111, |S|: ≈2^264.5, d: 911, prob: ≈2^-108.0, ↻: ≈2^110.2, tag: hybrid,
 'dual': rop: ≈2^149.9, mem: ≈2^97.1, m: 512, β: 424, d: 1024, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^139.7, red: ≈2^139.5, guess: ≈2^135.9, β: 387, p: 5, ζ: 0, t: 50, β': 391, N: ≈2^81.1, m: 512}

## Lattice estimator with our own parameters
Let's use Regev's methodology for choosing parameters; let's use base parameters similar to Kyber512

In [5]:
S = samples(m=512, n=512, lwe='Regev', seed=4815)

In [6]:
next_prime(pow(512, 2))

262147

In [40]:
from sage.stats.distributions.discrete_gaussian_integer import DiscreteGaussianDistributionIntegerSampler
# this comes very directly from sage.crypto.lwe.Regev


###### NOTE -- I don't think I have seen any paper use this “Both Gaussian” form. Trying just for funsies!
#choose n - nr of variables
n = 512
q = ZZ(next_prime(n**2))
sigma = RR(1/(RR(n).sqrt() * log(n, 2)**2) * q)
arith_mean = 0
# Chi-s - secret distribution
Xs = estimator.ND.DiscreteGaussian(sigma, arith_mean)
# Chi-e - error distribution
Xe = estimator.ND.DiscreteGaussian(sigma, arith_mean)
tag = 'Regev with Gaussian Xs and Gaussian Xe'

#optionally define how many samples we want to limit the oracle to
#sage.all.oo = infinity
m = oo

In [41]:
#attempt 0:
# estimator.lwe_parameters.LWEParameters(n, q, Xs, Xe, m, tag)
#got back:
# AttributeError: '...Sampler' object has no attribute 'resize'
#meaning we probably need to wrap sampler in a latticeEstimator compatible object (from package estimator.nd)

#attempt 1:
Regev_GaussXs_GaussXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_GaussXs_GaussXe

LWEParameters(n=512, q=262147, Xs=D(σ=143.03), Xe=D(σ=143.03), m=+Infinity, tag='Regev with Gaussian Xs and Gaussian Xe')

In [66]:
# the most standard form of Regev: uniform secret, Gaussian error
n = 512
q = ZZ(next_prime(n**2))
sigma = RR(1/(RR(n).sqrt() * log(n, 2)**2) * q)
arith_mean = 0
Xs = estimator.ND.UniformMod(q)
Xe = estimator.ND.DiscreteGaussian(sigma, arith_mean)
tag = 'Regev with uniform Xs and Gaussian Xe'
m = oo

Regev_UniXs_GaussXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_UniXs_GaussXe

LWEParameters(n=512, q=262147, Xs=D(σ=75675.32), Xe=D(σ=143.03), m=+Infinity, tag='Regev with uniform Xs and Gaussian Xe')

In [56]:
n = 512
q = ZZ(next_prime(n**2))
# Chi-s - secret distribution
Xs = estimator.ND.UniformMod(q)
# Chi-e - error distribution
##### manually chose modulus 496 as this would give parameter \sigma close to those with Gaussian distribution
Xe = estimator.ND.UniformMod(ZZ(495))
tag = 'Regev with uniform Xs and uniform Xe'
m = oo

Regev_UniXs_UniXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_UniXs_UniXe

LWEParameters(n=512, q=262147, Xs=D(σ=75675.32), Xe=D(σ=142.89), m=+Infinity, tag='Regev with uniform Xs and uniform Xe')

### Estimations on our own samples!

In [71]:
deny_list=["arora-gb", "bkw", "bdd", "bdd_hybrid", "bdd_mitm_hybrid", "dual"]

In [75]:
%%time
estimator.LWE.estimate(Regev_GaussXs_GaussXe, deny_list=deny_list)

usvp                 :: rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1571, tag: usvp
dual_hybrid          :: rop: ≈2^342.8, red: ≈2^342.2, guess: ≈2^341.3, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.8, m: 512
CPU times: user 677 ms, sys: 6.53 ms, total: 684 ms
Wall time: 692 ms


{'usvp': rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1571, tag: usvp,
 'dual_hybrid': rop: ≈2^342.8, red: ≈2^342.2, guess: ≈2^341.3, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.8, m: 512}

In [76]:
%%time
estimator.LWE.estimate(Regev_UniXs_GaussXe, deny_list=deny_list)

usvp                 :: rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1571, tag: usvp
dual_hybrid          :: rop: ≈2^342.8, red: ≈2^342.2, guess: ≈2^341.3, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.8, m: 512
CPU times: user 657 ms, sys: 2.88 ms, total: 660 ms
Wall time: 664 ms


{'usvp': rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1571, tag: usvp,
 'dual_hybrid': rop: ≈2^342.8, red: ≈2^342.2, guess: ≈2^341.3, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.8, m: 512}

In [77]:
%%time
estimator.LWE.estimate(Regev_UniXs_UniXe, deny_list=deny_list)

usvp                 :: rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1562, tag: usvp
dual_hybrid          :: rop: ≈2^342.7, red: ≈2^342.2, guess: ≈2^340.9, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.4, m: 512
CPU times: user 607 ms, sys: 767 μs, total: 608 ms
Wall time: 614 ms


{'usvp': rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1562, tag: usvp,
 'dual_hybrid': rop: ≈2^342.7, red: ≈2^342.2, guess: ≈2^340.9, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.4, m: 512}

It looks like sometimes the estimator can run for a lot-lot longer than expected. `UniXs_GaussXe` finished in 9.74 seconds wall time, while `UniXs_UniXe` did not estimate in a few minutes... Restarted kernel, then waited 24 minutes, did not finish. Swapped to `estimate.rough()` (from just `estimate()`).

---

Looking back, it was clear what the error was -- the choice of uniform distribution means that going for uniform errors in the entirety of $q$ would be ridiculously difficult. Thus, errors should probably be centered in some smaller subsection. I went back and modified the selection such that my errors would be from an odd modulus which would result a similar $\sigma$ value for the error distribution (in our case, original Gaussian $\chi_e$ had $\sigma\approx 143.03$; uniform $\chi_e$ with modulus $495$ gave $\sigma\approx 142.89$).